Purpose: st.Page acts as a blueprint or constructor to declare a page file/function, title, icon, and URL path for st.navigation. st.page_link creates a clickable UI element on the screen for users to jump to a target page or website

# app.py

What each part does:

set_page_config — must be the very first Streamlit call. Sets the browser tab title, favicon, and layout="wide" uses the full browser width instead of a narrow centered column.
st.Page — registers each page file. default=True makes Home the landing page.
st.navigation — builds the sidebar menu automatically from the page list. Returns whichever page the user selected.
nav.run() — executes the selected page's script.

# map_style.py



# map_config.py

##
The ** operator:

LEVELS["level_1"] returns a dict: {"grid_cols": 6, "grid_rows": 5}

** unpacks that dict into keyword arguments. So these two are identical:

python
# With **
get_map_config(**LEVELS["level_1"])

# Without ** (manually typing it out)
get_map_config(grid_cols=6, grid_rows=5)

It's a shortcut for passing a dict's contents as named arguments to a function.

## cpprd ans 
Why coord and extent were separate:

In your notebooks, map_coord controlled where the image was placed — its pixel corners mapped to world coordinates.

map_extent controlled the axis limits — what range of coordinates the plot shows.

They started the same, but remember when you needed to shift the image? You added x_offset and z_offset to map_coord to slide the image, but kept map_extent unchanged so the axis didn't move.

In the config, they start as the same value. If you need to shift the image again, you'd adjust coord without touching extent:

```python
# coord shifts the image
# extent stays fixed as the viewing window
````

If you never ended up needing that shift, we can simplify to just one value. Did you keep an offset in your final notebook, or did the image align without one?

## data_loader.py

```bash
example_game_data.json
├── saveDateTime, playerId                    → metadata
├── playerSaveObject
│   └── playerMovementSaveObject
│       └── playerMovementInfoList (185)      → player_movement_df
├── challengesSaveObject
│   └── challengesList[0]
│       ├── challengeId, challengeDuration... → challenge metadata
│       └── attemptsList (7)                  → challenge_df
├── gameEventsManagerSaveObject
│   └── gameEventsList (6)                    → game_events_df
└── validationManagerSaveObject
    └── playerValidationsList (3)             → validations_df
````



## data_export.py

In Streamlit, we won't write to disk — we'll use st.download_button with the CSV as bytes. So add a companion function:

Streamlit's download button doesn't work with file paths — it needs the file content as bytes in memory:

```python
def dataframe_to_csv_bytes(df: pd.DataFrame) -> bytes:
    """Convert a dataframe to CSV bytes for download."""
    return df.to_csv(index=False).encode("utf-8")
```
Usage in Streamlit later:

```python
csv_bytes = dataframe_to_csv_bytes(game_data["player_movement_df"])
st.download_button("Download Player Movement CSV", csv_bytes,
                   file_name="player_movement.csv", mime="text/csv")
```

**Streamlit download button expects bytes or string**
Streamlit's download button doesn't work with file paths — it needs the file content as bytes in memory:
```python
st.download_button(
    label="Download CSV",
    data=b"these are bytes",   # ← must be bytes
    file_name="export.csv"
)
```

So we need a way to convert a dataframe into bytes without saving to disk first:

**This writes to disk — works locally, not in cloud hosting
**df.to_csv("output.csv")

**This creates bytes in memory — works everywhere**
csv_bytes = df.to_csv(index=False).encode("utf-8")

# Learning

## Coding
It's a data engineering term — "data enrichment" means taking a base dataset and adding information from other sources to make it more useful. So player_movement_df (positions) gets "enriched" with challenge data (attempt numbers, states).

It's accurate, but jargon. If you haven't heard it before, your users/collaborators probably won't either. TIMELINE says what you get. ENRICHMENT says how you got it. For readable code, what-you-get wins.

## General Python

### .apply()
`.apply()` is a pandas method that runs a function on every row of a column:

```python
# This runs get_challenge_value_for_time once per row, 185 times
df["attempt_number"] = df["time"].apply(
    lambda t: get_challenge_value_for_time(challenge_df, t, "attempt_number")
)
```

vs doing it with a loop:

```python
# Same result, but manual
for i, row in df.iterrows():
    df.loc[i, "attempt_number"] = get_challenge_value_for_time(challenge_df, row["time"], "attempt_number")

.apply() is the pandas way — cleaner, slightly faster.
```

### np.integer and np.float
  return int(value) if isinstance(value, np.integer) else value
Examined type conversion logic and reconsidered function naming conventions
Examined type conversion logic and reconsidered function naming conventions

The line:

It's a one-liner if/else:

```python
# Same thing written out:
if isinstance(value, np.integer):
    return int(value)      # convert np.int64 → plain Python int
else:
    return value           # strings, floats — return as-is
```

Remember the np.int64(2) problem you hit earlier? This is the fix baked in. Pandas stores integers as numpy types. This line catches them and converts to plain Python int so you never get np.int64 leaking into your code.

##

      valid = [
            (pos, row)
            for pos, (_, row) in zip(positions, verb_df.iterrows())
            if pos is not None
        ]
Prepared to deconstruct list comprehension mechanics
Prepared to deconstruct list comprehension mechanics

Let's unpack it. Written as a loop:

python
valid = []
for pos, (_, row) in zip(positions, verb_df.iterrows()):
    if pos is not None:
        valid.append((pos, row))

Now piece by piece:

verb_df.iterrows() — yields (index, row) for each row in the dataframe. You get tuples like (0, Series(...)), (1, Series(...)).

positions — a list of (x, z) tuples or None values, one per event. Built from get_player_pos_for_time.

zip(positions, verb_df.iterrows()) — pairs them up side by side:

```python
# position           iterrows result
# (90.0, 60.0)       (0, Series(time=20.07, actor="dragon_Fog", ...))
# None               (1, Series(time=66.43, actor="player", ...))
# (84.2, 60.0)       (2, Series(time=81.39, actor="dragon_Tele...", ...))

for pos, (_, row) — unpacks each pair. pos gets the position tuple. (_, row) unpacks the iterrows result — _ throws away the index (we don't need it), row keeps the data.

if pos is not None — filters out events where we couldn't find a player position.

(pos, row) — keeps both pieces together as a tuple so we can use pos for coordinates and row for hover text later.
````

The result is a list of (position, row) pairs, only for events that have a valid player position.

## __init__
__init__.py is an empty file that tells Python "this folder is a package you can import from."

Without it:
```
modules/
├── data_loader.py
├── map_builder.py
```
Python sees modules/ as just a folder. from modules.data_loader import ... fails.

With it:
```
modules/
├── __init__.py          ← "I'm a package"
├── data_loader.py
├── map_builder.py
```
Python sees modules/ as a package. Imports work.

The touch command creates an empty file:

bash
touch modules/__init__.py
# Same as creating a blank file called __init__.py inside modules/

It stays empty — its existence is the signal, not its contents. Every folder you want to import from needs one.

#

## speed changes
2. Speed change during play — this is a plotly limitation. The workaround is to stop the current animation first. Unfortunately plotly doesn't support changing speed mid-play. The user has to click ⏸ then click the new speed. Add a note in the UI later, or accept it — it's a known constraint.

# video_player.py

Key design choices:

Class, not functions — because cv2.VideoCapture holds an open file handle. A class wraps the lifecycle: open in __init__, use with get_frame_*, close with release().
get_frame_at_time — this is what the synced page will call. Pass game time + offset, get the right video frame.
get_frame_at_index — for raw scrubbing when you don't need time mapping.
BGR→RGB conversion — done inside so callers never deal with OpenCV's color quirk.
Offset built in — no separate math needed in the page code.

Usage on the video sync page will look like:

```python
player = VideoPlayer("path/to/video.mp4")
frame = player.get_frame_at_time(game_time, offset=15.0)
st.image(frame)
player.release()
```

# sidebar_py

Key concepts:

render_sidebar() returns a dict with everything every page needs. Pages call this once and use the result.
_save_upload — Streamlit's file_uploader returns bytes in memory, not a file path. Our loaders expect file paths. This bridges the gap by saving to a temp folder.
Defaults — if no file is uploaded, it uses the example files. The app works out of the box.
st.sidebar — puts everything in the left panel, not the main content area.

# export.py

Simple wrappers — each page just calls csv_download_button(df, "filename.csv") or image_download_button(fig). No duplicate code across pages.

Now let's add __init__.py for the components folder:
touch creates an empty file. If the file already exists, it updates its timestamp without changing its content.

# Streamline pages

## home.py

Key Streamlit concepts:

st.columns(3) — creates three side-by-side columns. with col1: puts content in the first one.
st.metric — shows a value with a label, styled as a KPI card.
st.expander — collapsible section. Keeps the page clean, data available on click.
st.dataframe — renders a pandas dataframe as an interactive table.
render_sidebar() — called once, returns everything. The sidebar appears on every page because every page calls it.

## static_map.py

Key concepts:

st.plotly_chart(fig, use_container_width=True) — renders the plotly figure. use_container_width=True stretches it to fill the page width instead of using the fixed 800px from the figure layout.
Export buttons below the chart — PNG and CSV downloads right where the user is looking at the data.
Title uses actual data — player ID and challenge ID from the loaded files.

Test it — navigate to Static Map in the sidebar. You should see the full map with all traces, legend toggles, and three download buttons below. Let me know if anything is off.

fixedrange=True prevents zoom/pan from changing the axis range. Removing use_container_width=True keeps the figure at its defined 800x700. The map stays fixed regardless of browser window size.

python
fig.update_layout(
    width=800,
    height=700,
    autosize=False,
    xaxis=dict(
        range=[axis_range.x_min, axis_range.x_max],
        dtick=map_config["block_size"],
        showgrid=True,
        fixedrange=True,
        constrain="domain",
    ),
    yaxis=dict(
        range=[axis_range.z_min, axis_range.z_max],
        dtick=map_config["block_size"],
        showgrid=True,
        scaleanchor="x",
        fixedrange=True,
        constrain="domain",
    ),
)

constrain="domain" tells plotly "keep the axis range I set, shrink the plot area instead of expanding the range." This is the piece that was missing — without it, plotly fills extra space by extending the axis range.

## animated_map

Two-step flow: click "Export MP4" → spinner shows while rendering → download button appears when done. This avoids rendering 185 frames on every page load. It'll be slow (~1-2 minutes) but works. Test it and let me know.

### MP4 export

The key fix: frame_data = list(base_data) makes a copy of the trace list, then we replace specific indices with the frame's data. A fresh go.Figure(data=frame_data, layout=layout) avoids any in-place update issues.

**Why so slow**
Why so slow vs Jupyter:
In Jupyter, FuncAnimation updates pixels in memory — matplotlib swaps arrays directly. Here, each of your 185 frames goes through: Python → kaleido (headless browser) → render HTML/SVG → rasterize to PNG → decode bytes → OpenCV → write to video. That's a full browser render cycle per frame.

**session state**
Streamlit reruns your entire page script every time the user interacts with anything — clicks a button, moves a slider, even hovers. Without session_state, the video bytes would vanish on the next rerun.

```python
# Step 1: after export finishes, store the bytes
st.session_state["exported_video"] = video_bytes
# session_state is a dict that survives reruns (but not page changes)

# Step 2: on every rerun, check if the video exists
if "exported_video" in st.session_state:
    # It does — show the download button with the stored bytes
    st.download_button(...)
```

Without this, the flow would be: click export → video renders → download button appears → user clicks literally anything else on the page → script reruns → video bytes gone → download button disappears.

With session_state: click export → video renders → stored → download button appears → page reruns → video still in storage → download button stays.

**speed mismatch**


## video_sync.py

Key concepts:

st.stop() — halts the page if no video is uploaded. Nothing below runs.
st.columns(2) — video left, map right, side by side.
st.slider — every move reruns the script, which rebuilds the map and fetches a new video frame. Simple but works.
st.image — displays the OpenCV frame directly. No matplotlib needed.
The offset — sits in the sidebar. Type in how many seconds into the video the game starts.

The tradeoff: rebuilding the plotly figure on every slider move will feel sluggish. If it's too slow, we can cache the base figure and only rebuild the animated traces. Test it first and tell me how it performs.

This is the trickiest page. The challenge: Streamlit reruns the entire script on every slider move. We can't use plotly's built-in animation here — we need a Streamlit slider controlling both the video frame and a static plotly figure.

The approach:

Slider controls game time
Game time → video frame via offset
Game time → static plotly figure showing state at that moment
Both display side by side

First, let's add a helper to map_builder.py that builds a snapshot at a specific time:

### build frames cs get_frame_snapshot ###
build_frames — creates the animation data. For each of 185 time steps, it builds the changing traces (attempt paths growing, events appearing, player position, triangle). These get stored as fig.frames — plotly's animation system cycles through them during playback. The base traces (POIs, trail, map) stay untouched. Frames only contain what changes.

get_frame_snapshot — reads one frame and produces a complete static picture. It takes the base traces (POIs, trail, map) and merges in one frame's data (the player position, paths, events at that moment). Result is a standalone figure with no animation — just a screenshot of that instant.

Think of it like a film reel:

build_frames:         creates all 185 slides of the film
get_frame_snapshot:   pulls out slide #47 and prints it as a photo

build_frames runs once when building the animated figure. get_frame_snapshot runs every time the user moves the slider on the video sync page — it's cheap because it's just merging dicts, not recomputing anything.

###
The if "animated_fig" not in st.session_state block runs once — first visit to the page. After that, every slider move skips the build and just calls get_frame_snapshot, which is fast.

That's the core problem — plotly's animation slider doesn't talk back to Streamlit. When you scrub or play the plotly chart, Streamlit has no idea what frame it's on. There's no callback, no event, no way to read the current position.

Three options:

A. Drop plotly animation, use Streamlit slider for everything — back to snapshots. One slider controls both sides. No smooth playback, but perfect sync.

B. Keep plotly animation, accept manual input — what we just built. Animation plays smoothly, type the time to check the video.

C. Custom JavaScript bridge — embed JavaScript that reads plotly's slider value and sends it to Streamlit. This is complex but doable. Would look seamless to the user.

Which direction do you want? A is simplest and gives you exactly what you described. C is the most work but the best experience.